<a href="https://colab.research.google.com/github/luissosatorrentconsulting/TestingRepoMuni/blob/Historial-Empleados/ImportExportObjectSF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1: Global Parameters & Environment Setup
This module centralizes the cross-org connection parameters, migration switches, and dynamic target key mappings.

In [ ]:
# Install required dependencies directly into the Colab runtime container
!pip install simple-salesforce
!pip install numpy
!pip install pandas

import gc
import io
import os
import re
import sys
import time
from datetime import datetime
import numpy as np
import pandas as pd
from simple_salesforce import Salesforce
from google.colab import userdata

# --- GLOBAL CONFIGURATION VARIABLES (PARAMETRIZED) ---
# Set the variable equal to the API Name of the object being imported
apivariable = "Apttus_Config2__OrderLineItem__c"

# Core External ID field API Name used as the universal matching key
external_id_field = "Legacy_GSI_Id__c"

# Target Org active User ID to route records if Owner/Creator mappings fail
fallback_user_id = "005Kj00000CeE2RIAV"

# Control flags preserved from original design logic
Has_RecordTypes = False
Is_Detail = False
rerun = False
sortbyfield = ''
skip_existing = True
exclude_sales_restriction = False
autonumber_name = True

# Filter boundaries (Salesforce SOQL DateTime Formats: YYYY-MM-DDTHH:MM:SSZ)
use_date_filters = True
query_before_created_date = "2025-10-31T23:59:59Z"
query_after_modified_date = "2025-10-01T00:00:00Z"

# Define the number of chunks to split the processing dataset into
num_chunks = 10

# --- ENVIRONMENT CONNECTIONS ---


target_sf = Salesforce(
    username=userdata.get('username'),
    password=userdata.get('password'),
    consumer_key=userdata.get('consumer_key'),
    consumer_secret=userdata.get('consumer_secret'),
    domain="test"
)



source_sf = Salesforce(
    username="stephanie.poorman@torrentconsulting.com.oce",
    password="gqx4vrv7RTB7pdc.djw",
    security_token="P8yGaN0zHbXwssJtYBWBcOM9",
    domain="test"
)



# Shared statistics storage string instance
import_results = f"{apivariable} \n"
print(f"[{datetime.now()}] Independent notebook parameters initialized and active sessions cached.")

[2026-06-11 17:16:12.055495] Independent notebook parameters initialized and active sessions cached.


# Module 2: Asynchronous Bulk Query 2.0 Extraction
Leverages the native Salesforce Bulk 2.0 engine to handle high-volume extractions asynchronously on the Source environment, exporting results directly to an isolated local text stream asset.

In [ ]:
print(f"[{datetime.now()}] Launching asynchronous extraction job on Source Salesforce end...")

# Constructing SOQL statement mimicking the original database selection scope
soql_query = f"SELECT FIELDS(ALL) FROM {apivariable}"
if use_date_filters:
    soql_query += f" WHERE CreatedDate <= {query_before_created_date} AND LastModifiedDate >= {query_after_modified_date}"
soql_query += " LIMIT 1000000"

# Registering query transaction to Source Bulk Engine framework instance
query_job = source_sf.bulk2.__getattr__(apivariable).query(soql_query)
job_id = query_job['id']
print(f"Job registered to grid. Job ID: {job_id}. Polling server-side execution status...")

# Tracking execution state changes asynchronously
while True:
    job_status = source_sf.bulk2.__getattr__(apivariable).get_job_status(job_id)
    state = job_status['state']
    print(f"Current Job State: {state}")
    if state == 'JobComplete':
        break
    elif state in ['Failed', 'Aborted']:
        print("Fatal Exception: Source extraction crashed on backend grid. Halting notebook sequence.")
        sys.exit(1)
    time.sleep(15)

# Downloading database payload directly into an isolated local CSV structure
raw_source_csv_path = "source_extracted_raw.csv"
source_sf.bulk2.__getattr__(apivariable).get_elevated_results(job_id, raw_source_csv_path)
print(f"Success: Source rows extracted and physically stored at: {raw_source_csv_path}")